In [1]:
from tools.weather_tools.db import get_weather_data
import pandas as pd

In [2]:
df1 = pd.DataFrame(get_weather_data(pincode="686507"))
df1

,date,PRECTOTCORR,T2M,T2M_MAX,T2M_MIN,RH2M,WS2M,WS2M_MAX,ALLSKY_SFC_SW_DWN,T2MDEW,...,TSOIL6,PRECSNO,WS10M,WS10M_MAX,WS10M_MIN,WS10M_RANGE,WS50M,WS50M_MAX,WS50M_MIN,WS50M_RANGE
0,2006-01-01,0.00,21.54,26.57,17.26,78.25,0.45,0.92,21.08,17.32,...,24.19,0.0,1.22,2.21,0.08,2.13,2.18,3.81,0.19,3.62
1,2006-01-02,0.00,21.44,26.63,16.86,80.61,0.41,0.85,19.51,17.64,...,24.19,0.0,1.11,2.00,0.07,1.93,2.04,3.30,0.20,3.10
2,2006-01-03,0.00,21.94,26.73,17.74,80.18,0.40,0.72,19.72,18.11,...,24.19,0.0,1.03,1.78,0.09,1.69,1.87,3.32,0.41,2.91
3,2006-01-04,0.02,22.77,27.40,18.57,83.30,0.35,0.56,19.00,19.59,...,24.19,0.0,0.93,1.51,0.45,1.06,1.70,2.89,0.79,2.10
4,2006-01-05,0.28,23.46,27.16,20.19,86.56,0.32,0.48,17.36,20.97,...,24.19,0.0,0.79,1.25,0.16,1.09,1.37,2.03,0.41,1.62
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7519,2026-08-03,33.60,23.14,24.71,22.54,94.10,1.35,1.54,NaN,22.11,...,24.24,0.0,1.61,1.80,1.37,0.43,2.80,3.23,2.45,0.78
7520,2026-08-04,10.83,23.43,25.75,22.54,92.11,1.31,1.80,NaN,22.02,...,24.25,0.0,1.56,2.10,1.22,0.88,2.69,3.60,2.21,1.39
7521,2026-08-05,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7522,2026-08-06,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [3]:
import datetime
from collections import Counter

# --- thresholds -- judgment calls, documented so they're easy to revisit; the rain ones were
# validated against real data (see the markdown cell above this one for the threshold/window
# comparison across thresh1_win5 / thresh2.5_win5 / thresh5_win2 etc that led to these defaults) ---
RAIN_THRESHOLD_MM = 5.0        # raised from WMO's 1.0mm "wet day" definition -- trace/drizzle
                                # isn't what a farmer means by "will it rain"; 5mm is closer to
                                # IMD's "light rain" floor. Confirmed on real data: sharper
                                # wet/dry season contrast than a flat multi-year average
DAY_WINDOW_DEFAULT = 2          # +/-2 days pooled per calendar day (5-day pool, ~50 samples over
                                # 10yr) -- narrow enough to avoid over-smoothing real transitions
RAIN_CHANCE_LOW = 30            # % below this: "low chance" in the rain highlight
RAIN_CHANCE_HIGH = 60           # % at/above this: "high chance"; between = "moderate chance"

WIND_RISK_THRESHOLD_MS = 5.5    # ~20 km/h -- commonly cited spray-drift caution threshold
WIND_RISK_LOW = 20              # % below this: not called out in the wind highlight at all
WIND_RISK_HIGH = 50             # % at/above this: "high" wind risk; between = "moderate"

HEAT_STRESS_WETBULB_C = 28.0    # sustained wet-bulb temps at/above this are physiologically
                                 # stressful for outdoor labor (35C wet-bulb = often-cited human
                                 # survivability limit even in shade with unlimited water)
HUMID_FEEL_GAP_C = 4.0          # dry-bulb minus wet-bulb this close = high humidity, feels muggy
SOLAR_POOR_MJ = 10.0            # below this: poor/overcast for sun-drying (DATA_DICTIONARY.md)
SOLAR_GOOD_MJ = 18.0            # at/above this: good/strong sun for drying

HIGHLIGHT_WEEKLY_CUTOVER_DAYS = 14  # periods longer than this: highlights group by week-chunk
                                     # instead of specific days (a month-long ask shouldn't get
                                     # 30 individual day-clauses in a summary meant to be compact)


def _rain_category(mm: float) -> str:
    # IMD's 24-hour rainfall intensity classification, collapsed to plainer-language buckets
    if mm < 2.5:
        return "very light rain"
    if mm < 7.6:
        return "light rain"
    if mm < 35.6:
        return "moderate rain"
    if mm < 64.5:
        return "heavy rain"
    return "very heavy rain"


def _date_ranges(dates: list[datetime.date]) -> str:
    """[Aug9,Aug10,Aug11,Aug14] -> 'Aug 9-11, Aug 14'"""
    dates = sorted(dates)
    ranges, start, prev = [], dates[0], dates[0]
    for d in dates[1:]:
        if (d - prev).days == 1:
            prev = d
            continue
        ranges.append((start, prev))
        start = prev = d
    ranges.append((start, prev))

    def fmt(a, b):
        if a == b:
            return a.strftime("%b %-d")
        if a.month == b.month:
            return f"{a.strftime('%b %-d')}-{b.strftime('%-d')}"
        return f"{a.strftime('%b %-d')}-{b.strftime('%b %-d')}"

    return ", ".join(fmt(a, b) for a, b in ranges)


def _merge_consecutive(items: list, key_of) -> list[tuple[list, object]]:
    """Group strictly-consecutive (by list position) items sharing the same key_of(item) value."""
    groups, cur_items, cur_key = [], [], object()
    for item in items:
        k = key_of(item)
        if cur_items and k == cur_key:
            cur_items.append(item)
        else:
            if cur_items:
                groups.append((cur_items, cur_key))
            cur_items, cur_key = [item], k
    if cur_items:
        groups.append((cur_items, cur_key))
    return groups


def get_weather_forecast(
    lat: float | None = None,
    lon: float | None = None,
    pincode: str | None = None,
    district: str | None = None,
    state: str | None = None,
    start_date: str | None = None,
    period: int = 30,
    years_history: int = 10,
    day_window: int = DAY_WINDOW_DEFAULT,
) -> dict:
    """
    Get a climatological weather "forecast" for a location, as plain-language summary text --
    not a live forecast (this project only has NASA POWER's historical reanalysis, no forecast
    API); this reports what's typical for the time of year, from the last `years_history` years.

    v4: returns ONLY four short narrative strings (rain, comfort, wind, sunlight) -- no more
    per-day/week numeric table. An agent consuming this shouldn't have to scan a table and
    reason over raw numbers to answer "should I spray Thursday" -- it should get text that
    already says so. Every number that matters (rain %, rain amount in IMD-style plain
    language, feels-like temperature, wind risk %) is folded into its highlight sentence
    instead of sitting in a separate structure the agent has to cross-reference.

    Each highlight is ONE chunk per contiguous stretch of similar conditions, not repeated per
    day -- specific day-ranges ("Aug 9-11") for periods up to HIGHLIGHT_WEEKLY_CUTOVER_DAYS (14
    days), then week-chunks ("weeks of Aug 9, Aug 23") beyond that, so a month-long ask gets a
    handful of clauses, not ~30 near-duplicate daily ones. A stretch with no real day-to-day
    change in conditions collapses to one clause; a genuine transition (e.g. monsoon onset
    mid-window) shows up as two clauses with different numbers -- not smoothed into one average
    that would hide the transition.

    rain_chance_pct is a real historical frequency (RAIN_THRESHOLD_MM=5.0mm, +/-2 day pooling
    window -- both tuned against real data for a sharper wet/dry contrast, not squashed toward
    0/100 to look more "decisive," which would misrepresent the underlying frequency).

    Args:
        lat/lon, pincode, district, state: exactly one location mode, same as get_weather_data.
        start_date (str | None): first forecast day ('YYYY-MM-DD'). Defaults to tomorrow.
        period (int): number of days to forecast.
        years_history (int): years of history to draw from (default 10).
        day_window (int): +/- days pooled around each calendar day for the historical sample.

    Returns {"rain": str, "comfort": str, "wind": str, "sunlight": str}.
    """
    today = datetime.date.today()
    start = datetime.date.fromisoformat(start_date) if start_date else today + datetime.timedelta(days=1)
    history_start = datetime.date(today.year - years_history, 1, 1)  # calendar-year aligned

    raw = get_weather_data(
        lat=lat, lon=lon, pincode=pincode, district=district, state=state,
        start_date=history_start.isoformat(), end_date=today.isoformat(),
    )
    if not raw:
        raise ValueError("No historical data found for this location -- check the location resolves to a known grid cell.")

    hist = pd.DataFrame(raw)
    hist["date"] = pd.to_datetime(hist["date"])
    hist["doy"] = hist["date"].dt.dayofyear

    def _window_mask(target_doy: int) -> pd.Series:
        lo, hi = target_doy - day_window, target_doy + day_window
        if lo < 1:
            return (hist["doy"] >= lo + 366) | (hist["doy"] <= hi)
        if hi > 366:
            return (hist["doy"] >= lo) | (hist["doy"] <= hi - 366)
        return (hist["doy"] >= lo) & (hist["doy"] <= hi)

    def day_stats(d: datetime.date) -> dict:
        window = hist[_window_mask(d.timetuple().tm_yday)]

        rain = window["PRECTOTCORR"].dropna()
        wet_days = rain[rain >= RAIN_THRESHOLD_MM]
        rain_chance_pct = round(100 * len(wet_days) / len(rain)) if len(rain) else None
        rain_amount_mm = round(float(wet_days.mean()), 1) if len(wet_days) else 0.0

        wind = window["WS2M_MAX"].dropna()
        wind_risk_pct = round(100 * (wind >= WIND_RISK_THRESHOLD_MS).sum() / len(wind)) if len(wind) else None

        tmax = window["T2M_MAX"].mean()
        wetbulb = window["T2MWET"].mean()
        solar = window["ALLSKY_SFC_SW_DWN"].mean()

        heat_flag = bool(pd.notna(wetbulb) and wetbulb >= HEAT_STRESS_WETBULB_C)
        humid_flag = bool(pd.notna(wetbulb) and pd.notna(tmax) and (tmax - wetbulb) <= HUMID_FEEL_GAP_C)

        if pd.isna(solar):
            solar_cat = None
        elif solar < SOLAR_POOR_MJ:
            solar_cat = "poor"
        elif solar >= SOLAR_GOOD_MJ:
            solar_cat = "good"
        else:
            solar_cat = "moderate"

        return dict(
            rain_chance_pct=rain_chance_pct, rain_amount_mm=rain_amount_mm,
            feels_like_c=round(float(wetbulb), 1) if pd.notna(wetbulb) else None,
            wind_risk_pct=wind_risk_pct,
            heat_flag=heat_flag, humid_flag=humid_flag, solar_cat=solar_cat,
        )

    forecast_dates = [start + datetime.timedelta(days=i) for i in range(period)]
    per_day = {d: day_stats(d) for d in forecast_dates}

    use_weekly = period > HIGHLIGHT_WEEKLY_CUTOVER_DAYS
    chunks = [forecast_dates[i : i + 7] for i in range(0, len(forecast_dates), 7)] if use_weekly else [[d] for d in forecast_dates]

    def _label(flat_days: list[datetime.date], group_chunks: list[list[datetime.date]]) -> str:
        if use_weekly:
            return "weeks of " + ", ".join(c[0].strftime("%b %-d") for c in group_chunks)
        return _date_ranges(flat_days)

    # --- rain ---
    def chunk_rain_bucket(chunk):
        vals = [per_day[d]["rain_chance_pct"] for d in chunk if per_day[d]["rain_chance_pct"] is not None]
        if not vals:
            return None
        pct = round(sum(vals) / len(vals))
        return "low" if pct < RAIN_CHANCE_LOW else ("high" if pct >= RAIN_CHANCE_HIGH else "moderate")

    rain_parts = []
    for group_chunks, bucket in _merge_consecutive(chunks, chunk_rain_bucket):
        if bucket is None:
            continue
        flat = [d for c in group_chunks for d in c]
        pcts = [per_day[d]["rain_chance_pct"] for d in flat if per_day[d]["rain_chance_pct"] is not None]
        amts = [per_day[d]["rain_amount_mm"] for d in flat if per_day[d]["rain_amount_mm"]]
        avg_pct = round(sum(pcts) / len(pcts)) if pcts else 0
        avg_amt = round(sum(amts) / len(amts), 1) if amts else 0.0
        label = _label(flat, group_chunks)
        if bucket == "low":
            rain_parts.append(f"{label}: low chance of rain (~{avg_pct}%)")
        else:
            rain_parts.append(f"{label}: {bucket} chance of rain (~{avg_pct}%), expect {_rain_category(avg_amt)} (~{avg_amt}mm) when it falls")
    rain_highlight = ("; ".join(rain_parts) + ".") if rain_parts else "No significant rain chance expected."

    # --- comfort (now always includes the feels-like number, every group -- this is the only
    # place a temperature figure appears anywhere in the output) ---
    def chunk_comfort(chunk):
        if any(per_day[d]["heat_flag"] for d in chunk):
            return "stressful"
        if any(per_day[d]["humid_flag"] for d in chunk):
            return "humid"
        return "comfortable"

    comfort_parts = []
    for group_chunks, level in _merge_consecutive(chunks, chunk_comfort):
        flat = [d for c in group_chunks for d in c]
        feels = [per_day[d]["feels_like_c"] for d in flat if per_day[d]["feels_like_c"] is not None]
        avg_feels = round(sum(feels) / len(feels), 1) if feels else None
        feels_str = f"feels like ~{avg_feels}C" if avg_feels is not None else "no temperature data"
        label = _label(flat, group_chunks)
        if level == "stressful":
            comfort_parts.append(f"{label}: {feels_str}, humid heat-stress risk -- avoid strenuous outdoor work midday")
        elif level == "humid":
            comfort_parts.append(f"{label}: {feels_str}, high humidity -- will feel hotter/muggier than the temperature alone suggests")
        else:
            comfort_parts.append(f"{label}: {feels_str}, comfortable humidity")
    comfort_highlight = "; ".join(comfort_parts) + "."

    # --- wind (only called out when actually elevated -- "low" is the expected default state,
    # unlike rain/sunlight where every period gets a stated level) ---
    def chunk_wind_level(chunk):
        vals = [per_day[d]["wind_risk_pct"] for d in chunk if per_day[d]["wind_risk_pct"] is not None]
        if not vals:
            return "unknown"
        pct = round(sum(vals) / len(vals))
        if pct >= WIND_RISK_HIGH:
            return "high"
        if pct >= WIND_RISK_LOW:
            return "moderate"
        return "low"

    wind_parts = []
    for group_chunks, level in _merge_consecutive(chunks, chunk_wind_level):
        if level in ("low", "unknown"):
            continue
        flat = [d for c in group_chunks for d in c]
        pcts = [per_day[d]["wind_risk_pct"] for d in flat if per_day[d]["wind_risk_pct"] is not None]
        avg_pct = round(sum(pcts) / len(pcts)) if pcts else 0
        label = _label(flat, group_chunks)
        wind_parts.append(f"{label}: {level} wind risk (~{avg_pct}%) -- caution advised for spraying")
    wind_highlight = ("; ".join(wind_parts) + ".") if wind_parts else "Low wind risk throughout -- good conditions for spraying."

    # --- sunlight ---
    def chunk_solar(chunk):
        cats = [per_day[d]["solar_cat"] for d in chunk if per_day[d]["solar_cat"]]
        return Counter(cats).most_common(1)[0][0] if cats else None

    solar_parts = []
    for group_chunks, cat in _merge_consecutive(chunks, chunk_solar):
        if cat is None:
            continue
        flat = [d for c in group_chunks for d in c]
        label = _label(flat, group_chunks)
        if cat == "good":
            solar_parts.append(f"{label}: good sun for drying")
        elif cat == "poor":
            solar_parts.append(f"{label}: mostly cloudy, poor sun-drying conditions")
        else:
            solar_parts.append(f"{label}: moderate sunlight")
    sunlight_highlight = ("; ".join(solar_parts) + ".") if solar_parts else "Not enough data to summarize sunlight."

    return {"rain": rain_highlight, "comfort": comfort_highlight, "wind": wind_highlight, "sunlight": sunlight_highlight}

### Test 1: 7-day forecast -- now just four narrative strings, no table

In [4]:
daily = get_weather_forecast(pincode="686507", period=7)
for k, v in daily.items():
    print(f"{k}: {v}")

rain: Aug 9-15: moderate chance of rain (~54%), expect moderate rain (~18.2mm) when it falls.
comfort: Aug 9-15: feels like ~22.9C, high humidity -- will feel hotter/muggier than the temperature alone suggests.
wind: Low wind risk throughout -- good conditions for spraying.
sunlight: Aug 9-15: moderate sunlight.


### Test 2: month-long forecast (28 days) -- highlights should switch to week-chunk grouping

In [5]:
monthly = get_weather_forecast(pincode="686507", period=28)
for k, v in monthly.items():
    print(f"{k}: {v}")

rain: weeks of Aug 9, Aug 16, Aug 23: moderate chance of rain (~47%), expect moderate rain (~15.4mm) when it falls; weeks of Aug 30: high chance of rain (~64%), expect moderate rain (~13.7mm) when it falls.
comfort: weeks of Aug 9, Aug 16, Aug 23, Aug 30: feels like ~23.0C, high humidity -- will feel hotter/muggier than the temperature alone suggests.
wind: Low wind risk throughout -- good conditions for spraying.
sunlight: weeks of Aug 9: moderate sunlight; weeks of Aug 16: good sun for drying; weeks of Aug 23, Aug 30: moderate sunlight.


### Why probability instead of averaging: same Kerala location, monsoon vs. dry season

Now checking the `highlights.rain` narrative directly, not raw per-day numbers -- this is what the agent would actually see.

In [6]:
monsoon = get_weather_forecast(pincode="686507", start_date="2027-07-01", period=7)
dry_season = get_weather_forecast(pincode="686507", start_date="2027-01-15", period=7)

print("=== Monsoon (early July) ===")
for k, v in monsoon.items():
    print(f"  {k}: {v}")

print("\n=== Dry season (mid January) ===")
for k, v in dry_season.items():
    print(f"  {k}: {v}")

=== Monsoon (early July) ===
  rain: Jul 1-3: high chance of rain (~63%), expect moderate rain (~14.2mm) when it falls; Jul 4-7: moderate chance of rain (~57%), expect moderate rain (~14.1mm) when it falls.
  comfort: Jul 1-7: feels like ~23.2C, high humidity -- will feel hotter/muggier than the temperature alone suggests.
  wind: Low wind risk throughout -- good conditions for spraying.
  sunlight: Jul 1-7: moderate sunlight.

=== Dry season (mid January) ===
  rain: Jan 15-21: low chance of rain (~5%).
  comfort: Jan 15-21: feels like ~20.7C, comfortable humidity.
  wind: Low wind risk throughout -- good conditions for spraying.
  sunlight: Jan 15-21: good sun for drying.


### Correctness check: verify the numbers embedded in the narrative text, plus a real wind-risk case

No more per-day table to check field-by-field -- verify by extracting the numbers embedded in each highlight string (regex) and comparing to a manual pandas computation. Also confirms `wind` actually fires (not just always "Low wind risk") against Kachchh, Gujarat in May -- the location/season already shown earlier in this project to have ~85% of days over the wind-risk threshold.

In [7]:
import re

check_df = df1.copy()
check_df["date"] = pd.to_datetime(check_df["date"])

today = datetime.date.today()
history_start = pd.Timestamp(today.year - 10, 1, 1)
check_df = check_df[(check_df["date"] >= history_start) & (check_df["date"] <= pd.Timestamp(today))]
check_df["doy"] = check_df["date"].dt.dayofyear

target = today + datetime.timedelta(days=1)  # first forecast day
target_doy = target.timetuple().tm_yday
lo, hi = target_doy - 2, target_doy + 2  # matches DAY_WINDOW_DEFAULT=2
mask = (check_df["doy"] >= lo) & (check_df["doy"] <= hi)
window = check_df[mask]

manual_wetbulb = round(float(window["T2MWET"].mean()), 1)
rain = window["PRECTOTCORR"].dropna()
wet_days = rain[rain >= 5.0]  # matches RAIN_THRESHOLD_MM=5.0
manual_rain_chance = round(100 * len(wet_days) / len(rain))
manual_bucket = "low" if manual_rain_chance < 30 else ("high" if manual_rain_chance >= 60 else "moderate")

print("comfort text:", daily["comfort"])
print("rain text:   ", daily["rain"])

comfort_feels = float(re.search(r"feels like ~([\d.]+)C", daily["comfort"]).group(1))
rain_pct = int(re.search(r"~(\d+)%", daily["rain"]).group(1))

print(f"\nmanual feels_like (single day): {manual_wetbulb}C  vs  comfort text (group avg): {comfort_feels}C")
print(f"manual rain_chance (single day): {manual_rain_chance}%  vs  rain text (group avg): {rain_pct}%, bucket={manual_bucket}")

assert abs(comfort_feels - manual_wetbulb) <= 1.0, "feels-like group average unexpectedly far from single-day value"
assert abs(rain_pct - manual_rain_chance) <= 15, "rain % group average unexpectedly far from single-day value"
assert manual_bucket in daily["rain"], "single day's bucket should appear in the rain highlight"
print("\nOK: embedded numbers are consistent with manual computation.")

# --- real wind-risk case: Kachchh, Gujarat, May -- shown earlier in this project to have
# ~85% of days over the wind-risk threshold in that month ---
windy = get_weather_forecast(district="kachchh", start_date="2027-05-01", period=7)
print("\nKachchh, May 1-7 wind highlight:", windy["wind"])
assert "Low wind risk" not in windy["wind"], "expected elevated wind risk to actually fire for a known-windy season"
print("OK: wind highlight correctly fires for a real high-wind case (not always silent).")

comfort text: Aug 9-15: feels like ~22.9C, high humidity -- will feel hotter/muggier than the temperature alone suggests.
rain text:    Aug 9-15: moderate chance of rain (~54%), expect moderate rain (~18.2mm) when it falls.

manual feels_like (single day): 22.9C  vs  comfort text (group avg): 22.9C
manual rain_chance (single day): 58%  vs  rain text (group avg): 54%, bucket=moderate

OK: embedded numbers are consistent with manual computation.



Kachchh, May 1-7 wind highlight: May 1-7: high wind risk (~74%) -- caution advised for spraying.
OK: wind highlight correctly fires for a real high-wind case (not always silent).


### "Minimal token in context" -- v4 (narrative-only) vs raw data

Rough proxy: JSON character count (~4 chars/token). Compares the 7-day forecast (now just 4 strings) against the equivalent raw `get_weather_data` slice, and shows the 28-day monthly forecast is barely bigger than the 7-day one -- the whole point of week-chunking once a period gets long.

In [8]:
import json

compact_json = json.dumps(daily)  # the 7-day, 4-string forecast from Test 1

raw_days = df1.copy()
raw_days["date"] = pd.to_datetime(raw_days["date"])
raw_window = raw_days[(raw_days["date"] >= pd.Timestamp(today) - pd.Timedelta(days=6)) & (raw_days["date"] <= pd.Timestamp(today))]
raw_json = raw_window.to_json(orient="records")

print(f"compact (4 narrative strings): {len(compact_json)} chars (~{len(compact_json)//4} tokens)")
print(f"raw equivalent:                 {len(raw_json)} chars (~{len(raw_json)//4} tokens), {len(raw_window)} days x {len(raw_days.columns)-1} columns")
print(f"compaction ratio: {len(raw_json) / len(compact_json):.1f}x smaller")

monthly_json = json.dumps(monthly)  # 28-day forecast from Test 2
print(f"\n28-day forecast: {len(monthly_json)} chars (~{len(monthly_json)//4} tokens) -- "
      f"only {len(monthly_json)/len(compact_json):.1f}x bigger than the 7-day one, for 4x the coverage")

compact (4 narrative strings): 334 chars (~83 tokens)
raw equivalent:                 3481 chars (~870 tokens), 6 days x 35 columns
compaction ratio: 10.4x smaller

28-day forecast: 565 chars (~141 tokens) -- only 1.7x bigger than the 7-day one, for 4x the coverage


## The real tools (gramsahayak/agent/tools/weather_tools/forecast_tools.py)

In [9]:
from tools.weather_tools.forecast_tools import (
    get_weather_forecast, query_spray_window, query_sowing_window,
    query_pest_pressure_window, query_crop_suitability,
)

In [10]:
get_weather_forecast(pincode="686507", start_date="2027-07-01", period=7)

{'rain': 'Jul 1-3: high chance of rain (~63%), expect moderate rain (~14.2mm) when it falls; Jul 4-7: moderate chance of rain (~57%), expect moderate rain (~14.1mm) when it falls.',
 'comfort': 'Jul 1-7: feels like ~23.2C, high humidity -- will feel hotter/muggier than the temperature alone suggests.',
 'wind': 'Low wind risk throughout.',
 'sunlight': 'Jul 1-7: moderate sunlight.',
 'flood': 'No significant flood risk expected.',
 'heat_wave': 'No heat-wave risk expected.'}

In [11]:
query_spray_window(pincode="686507", start_date="2027-01-15", period=14)

{'spray_window': 'Best spray window: Jan 15-28 (14 day(s), ~4% rain chance, ~0% wind risk).',
 'skip_watering': 'No significant rain expected in this period -- irrigation likely needed throughout.',
 'rain_protection_warning': 'No flood-level rain expected -- no special protection needed for rain-sensitive crops.'}

In [12]:
query_sowing_window(pincode="686507", water_need="wet", start_date="2027-05-01", months=3)

{'sowing_window': "Best sowing window for a 'wet'-preference crop: weeks of May 15, May 22, May 29, Jun 5. Soil moisture profile expected saturated -- monitor for waterlogging (GWETPROF ~0.7).",
 'flood_warning': 'No flood risk expected in this period.',
 'drought_warning': 'No extended dry-spell risk expected.'}

In [13]:
query_pest_pressure_window(district="chennai", start_date="2027-10-01", period=14)

{'pest_pressure': 'Oct 1-14: high pest-favorable conditions (~99% of years warm+humid) -- extra scouting recommended.',
 'caveat': 'Generic warm+humid heuristic, not a species-specific pest forecast.'}

In [14]:
query_crop_suitability(pincode="686507", water_need="dry", heat_sensitive=True, start_date="2027-07-01", months=1)

{'forecast': {'rain': 'weeks of Jul 1: moderate chance of rain (~59%), expect moderate rain (~14.2mm) when it falls; weeks of Jul 8, Jul 15: high chance of rain (~63%), expect moderate rain (~14.2mm) when it falls; weeks of Jul 22, Jul 29: moderate chance of rain (~56%), expect moderate rain (~13.9mm) when it falls.',
  'comfort': 'weeks of Jul 1, Jul 8, Jul 15, Jul 22, Jul 29: feels like ~23.1C, high humidity -- will feel hotter/muggier than the temperature alone suggests.',
  'wind': 'Low wind risk throughout.',
  'sunlight': 'weeks of Jul 1, Jul 8, Jul 15, Jul 22, Jul 29: moderate sunlight.',
  'flood': 'No significant flood risk expected.',
  'heat_wave': 'No heat-wave risk expected.'},
 'assessment_note': 'Weather outlook above covers the next 1 month(s) for this location. Assess suitability considering: this crop needs dry conditions -- weigh the rain outlook above as a risk, not a benefit; this crop is heat-sensitive -- treat the heat_wave/comfort outlook above as a direct risk 